Analytical Model

Architecture DOR:

- p - physical channel (0 - wide bus; 1 - narrow bus)
- v - virtual channel (0 - read/write request, YX routing; 1 - read/write response, XY routing)
- s - source tile
- d - destination tile
- r - routing tile
- i - input direction
- o - output direction

Routing Tensor (RT):
RT[s][d][p][v][r][i][o]

Packet injection rate (f):
f[s][d][p][v] - packet injection rate for route s -> d via physical channel p, virtual channel v:
- f[s][d][0][0] - write request (s - perimeter; d - internal)
- f[s][d][0][1] - read response (s - internal; d - perimeter)
- f[s][d][1][0] - read request (s - perimeter; d - internal)
- f[s][d][1][1] - write response (s - internal; d - perimeter)

Packet Length (L):
L[p][v]

Router packet Arrival rate (A):
A[p][v][r][i][o] = Sum{any s; any d}(f[s][d][p][v] * RT[s][d][p][v][r][i][o])

Input buffer Arrival rate (IA):
IA[p][v][r][i] = Sum{any o}(A[p][v][r][i][o])

Output buffer Arrival rate (OA):
OA[p][v][r][o] = Sum{any i}(A[p][v][r][i][o])

Waiting Time of body flits from other directions in Input buffer (Tbody):
Tbody[p][v][r][i][o] = Sum{any k != i}(A[p][v][r][k][o] / L[p][v] * Sum{q from 1 to L[p][v]}(L - q)) = (L[p][v] - 1) / 2 * Sum{any k != i}(A[p][v][r][k][o])



In [1]:
import numpy as np

In [2]:
PHYS_NUM = 2
VIRT_NUM = 2

NORTH_IDX = 0
SOUTH_IDX = 1
WEST_IDX = 2
EAST_IDX = 3
LOCAL_IDX = 4
DIR_NUM = 5

X_NUM = 10
Y_NUM = 10
TILE_NUM = X_NUM * Y_NUM

In [3]:
class Coord:
    def __init__(self, id=0):
        self.x = id % X_NUM
        self.y = id // Y_NUM

    def __repr__(self):
        return f"Coordinate(x={self.x}, y={self.y})"

    def is_perimeter(self):
        return (self.x == 0) or (self.x == X_NUM - 1) or (self.y == 0) or (self.y == Y_NUM - 1)

    def is_vertical(self):
        return (self.x == 0) or (self.x == X_NUM - 1)

    def is_horizontal(self):
        return self.is_perimeter() and not self.is_vertical()

    def id(self):
        return self.y * Y_NUM + self.x

In [4]:
# Returns next tile for given current and destination ones
# Algorith DOR
def next_tile_gen_yx(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    elif current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    return next

def next_tile_gen_xy(current_tile, dst_tile):
    next = Coord(current_tile.id())
    if current_tile.x < dst_tile.x:
        next.x += 1
    elif current_tile.x > dst_tile.x:
        next.x -= 1
    elif current_tile.y < dst_tile.y:
        next.y += 1
    elif current_tile.y > dst_tile.y:
        next.y -= 1
    return next

In [5]:
# Returns directions current_tile output ID and next_tile input ID
def encode_dirs(current_tile, next_tile):
    if current_tile.y < next_tile.y:
        return (SOUTH_IDX, NORTH_IDX)
    elif current_tile.y > next_tile.y:
        return (NORTH_IDX, SOUTH_IDX)
    elif current_tile.x < next_tile.x:
        return (EAST_IDX, WEST_IDX)
    elif current_tile.x > next_tile.x:
        return (WEST_IDX, EAST_IDX)
    return (LOCAL_IDX, LOCAL_IDX)

In [ ]:
def calc_collision_possibility_internal(request_possibility, i, o, c, num):
    # The four other indices besides i
    others = [x for x in range(num) if x != i]
    others_num = num - 1
    others_comb_num = 2**others_num
    total = 0.0
    # Loop over all 16 combinations of (k1, k2, k3, k4) in {0,1}^4
    for mask in range(others_comb_num):  # from 0 to 15
        # Count how many bits are 1
        # and build the product F[...]^(k_t) * (1-F[...])^(1-k_t)
        ksum = 0
        prob_product = 1.0
        for bit_idx in range(others_num):
            # k_t is either 0 or 1
            k_t = (mask >> bit_idx) & 1
            p = request_possibility[others[bit_idx]][o]
            if k_t == 1:
                prob_product *= p
                ksum += 1
            else:
                prob_product *= (1 - p)

        # If k_1 + k_2 + k_3 + k_4 = c, add to sum
        if ksum == c:
            total += prob_product
    return total


def calc_collision_possibility(request_possibility, i, o, num):
    blocking_possibility = 0.0
    for c in range(1, num):
        blocking_possibility_internal = calc_collision_possibility_internal(
            request_possibility, i, o, c, num)
        blocking_possibility += blocking_possibility_internal * \
            (c / (c + 1))
    return blocking_possibility


def solve_request_possibility(payload_tensor, flow_control_possibility, max_iter=1000, tol=1e-8):
    request_possibility = payload_tensor.copy()

    (input_num, output_num) = payload_tensor.shape

    for _ in range(max_iter):
        request_possibility_old = request_possibility.copy()

        for i in range(input_num):
            for o in range(output_num):
                collision_possibility = calc_collision_possibility(
                    request_possibility, i, o, input_num)
                blocking_possibility = 1.0 - (1.0 - collision_possibility) * (1.0 - flow_control_possibility[i][o])

                denom = 1.0 - blocking_possibility
                if abs(denom) < 1e-14:
                    # Avoid dividing by zero.
                    # Could set F[i][j] to some fallback value.
                    request_possibility[i][o] = 0.999999 if denom < 0 else 0.0
                else:
                    request_possibility[i][o] = payload_tensor[i][o] / denom

        # Check for convergence
        diff = np.linalg.norm(request_possibility - request_possibility_old)
        if diff < tol:
            break

    # print(f"Finished in {it+1} iterations with diff={diff}")
    return request_possibility


def calc_blocking_possibility(request_possibility, flow_control_possibility):
    blocking_possibility = request_possibility.copy()
    (input_num, output_num) = blocking_possibility

    for i in range(input_num):
        for o in range(output_num):
            collision_possibility = calc_collision_possibility(request_possibility, i, o, input_num)
            blocking_possibility[i][o] = 1.0 - (1.0 - collision_possibility) * (1.0 - flow_control_possibility[i][o])

    return blocking_possibility

In [6]:
#                          [p]       [v]       [s]       [d]       [r]       [i]      [o]
routing_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM, TILE_NUM, DIR_NUM, DIR_NUM))

def is_traffic_exist(s, d, is_forward):
    s_coord = Coord(s)
    d_coord = Coord(d)
    if is_forward and (s_coord.is_perimeter() and not d_coord.is_perimeter()):
        return True
    if not is_forward and (not s_coord.is_perimeter() and d_coord.is_perimeter()):
        return True
    return False

def src_dst_routing_tensor(s, d, is_forward):
    #              [r]       [i]      [o]
    rt = np.zeros((TILE_NUM, DIR_NUM, DIR_NUM))

    if not is_traffic_exist(s, d, is_forward):
        return rt

    s_coord = Coord(s)
    d_coord = Coord(d)
    current_tile = s_coord
    i_cur_dir = LOCAL_IDX
    while current_tile.id() != d:
        next_tile = next_tile_gen_yx(current_tile, d_coord) if is_forward else \
                    next_tile_gen_xy(current_tile, d_coord)
        (o_cur_dir, i_nxt_dir) = encode_dirs(current_tile, next_tile)
        rt[current_tile.id()][i_cur_dir][o_cur_dir] = 1.0
        current_tile = Coord(next_tile.id())
        i_cur_dir = i_nxt_dir
    rt[d][i_cur_dir][LOCAL_IDX] = 1.0

    return rt

for p in range(PHYS_NUM):
    for v in range(VIRT_NUM):
        for s in range(TILE_NUM):
            for d in range(TILE_NUM):
                is_forward = (v == 0)
                routing_tensor[p][v][s][d] = src_dst_routing_tensor(s, d, is_forward)


Routing tensor done


In [ ]:
#                                [p]       [v]
packet_length_tensor = np.zeros((PHYS_NUM, VIRT_NUM))

# Write request
packet_length_tensor[0][0] = 1.0
# Read response
packet_length_tensor[0][1] = 1.0
# Read request
packet_length_tensor[1][0] = 1.0
# Write response
packet_length_tensor[1][1] = 1.0

In [ ]:
def calc_packet_injection_rate_tensor(pir):
    #                                        [p]       [v]       [s]       [d]
    packet_injection_rate_tensor = np.zeros((PHYS_NUM, VIRT_NUM, TILE_NUM, TILE_NUM))

    for p in range(PHYS_NUM):
        for v in range(VIRT_NUM):
            for s in range(TILE_NUM):
                for d in range(TILE_NUM):
                    is_forward = (v == 0)
                    if is_traffic_exist(s, d, is_forward):
                        packet_injection_rate_tensor[p][v][s][d] = pir

    return packet_injection_rate_tensor

In [ ]:
def calc_router_packet_payload_tensor(routing_tensor, packet_injection_rate_tensor):
    multiplied = routing_tensor * packet_injection_rate_tensor[..., np.newaxis, np.newaxis, np.newaxis]
    router_packet_payload_tensor = multiplied.sum(axis=(2, 3))
    # [p] [v] [r] [i] [o]
    return router_packet_payload_tensor

In [11]:
def calc_router_input_packet_payload_tensor(router_packet_payload_tensor):
    # [p] [v] [r] [i]
    return router_packet_payload_tensor.sum(axis=-1)

In [12]:
def calc_router_output_packet_payload_tensor(router_packet_payload_tensor):
    # [p] [v] [r] [o]
    return router_packet_payload_tensor.sum(axis=3)

In [13]:
def calc_blocking_possibility_tensor(router_payload_tensor):
    blocking_possibility = np.zeros((tile_num, direction_num, direction_num))

    for r in range(tile_num):
        request_possibility_solution = solve_request_possibility(
            router_payload_tensor[r], max_iter=500, tol=1e-10)
        for i in range(direction_num):
            for o in range(direction_num):
                blocking_possibility[r][i][o] = calc_blocking_possibility(
                    request_possibility_solution, i, o)

    print("Blocking possibility done")
    return blocking_possibility

In [14]:
def calc_blocking_time(blocking_possibility):
    blocking_time = blocking_possibility.copy()

    for r in range(tile_num):
        for i in range(direction_num):
            for o in range(direction_num):
                blocking_time[r][i][o] /= (1 - blocking_time[r][i][o])

    print("Blocking time done")
    return blocking_time

In [15]:
def calc_request_handle_time(blocking_time):
    request_handle_time = blocking_time.copy()

    for r in range(tile_num):
        for i in range(direction_num):
            for o in range(direction_num):
                request_handle_time[r][i][o] += 1.

    print("Request handle time done")
    return request_handle_time

In [16]:
def calc_mean_input_handle_time(router_input_payload_tensor, router_payload_tensor, request_handle_time):
    mean_input_handle_time = np.zeros((tile_num, direction_num))

    for r in range(tile_num):
        for i in range(direction_num):
            sum_input_handle_time = 0.
            for o in range(direction_num):
                sum_input_handle_time += router_payload_tensor[r][i][o] * \
                    request_handle_time[r][i][o]
            mean_input_handle_time[r][i] = 0. if (abs(sum_input_handle_time) < 1e-9) else sum_input_handle_time / \
                router_input_payload_tensor[r][i]

    print("Mean request handle time done")
    return mean_input_handle_time

In [17]:
def mm1n_queue(lambda_arrival, E_T, N):
    utilization = lambda_arrival * E_T
    pk_values = [(1 - utilization) * utilization**k / (1 - utilization**(N+1))
                 for k in range(N+1)]
    mean_depth = sum((k - 1) * pk_values[k] for k in range(1, N+1))
    return mean_depth

In [18]:
def calc_mean_input_queue_depth(router_input_payload_tensor, mean_input_handle_time):
    mean_input_queue_depth = np.zeros((tile_num, direction_num))

    for r in range(tile_num):
        for i in range(direction_num):
            mean_input_queue_depth[r][i] = mm1n_queue(
                router_input_payload_tensor[r][i], mean_input_handle_time[r][i], 2)
    print("Mean queue depth done")
    return mean_input_queue_depth

In [19]:
def calc_average_latency(routing_tensor, mean_input_queue_depth, mean_input_handle_time):
    average_latency = 0.0
    for r in range(tile_num):
        for i in range(direction_num):
            if abs(routing_tensor[r][i].sum()) > 1e-9:
                average_latency += (mean_input_queue_depth[r][i] + 1.) * \
                    mean_input_handle_time[r][i] + 1.
    return average_latency

In [20]:
def calc_average_latency_tensor(routing_tensor, mean_input_queue_depth, mean_input_handle_time):
    average_latency = np.zeros((tile_num, tile_num))

    for s in range(tile_num):
        for d in range(tile_num):
            if abs(routing_tensor[s][d].sum()) > 1e-9:
                average_latency[s][d] = calc_average_latency(
                    routing_tensor[s][d], mean_input_queue_depth, mean_input_handle_time)

                # print(Coord(s), Coord(d), average_latency[s][d])

    print("Average latency done")
    return average_latency

In [21]:
def calc_average_latency_pir_pipeline(pir):
  pir_per_mem_tile = pir / (mesh_dim_x - 2) / (mesh_dim_y - 2)
  router_req_payload_tensor_copy = router_req_payload_tensor.copy() * pir_per_mem_tile
  router_resp_payload_tensor_copy = router_resp_payload_tensor.copy() * pir_per_mem_tile
  router_req_input_payload_tensor = calc_router_input_payload_tensor(router_req_payload_tensor_copy)
  router_resp_input_payload_tensor = calc_router_input_payload_tensor(router_resp_payload_tensor_copy)
  req_blocking_possibility = calc_blocking_possibility_tensor(router_req_payload_tensor_copy)
  resp_blocking_possibility = calc_blocking_possibility_tensor(router_resp_payload_tensor_copy)
  req_blocking_time = calc_blocking_time(req_blocking_possibility)
  resp_blocking_time = calc_blocking_time(resp_blocking_possibility)
  req_handle_time = calc_request_handle_time(req_blocking_time)
  resp_handle_time = calc_request_handle_time(resp_blocking_time)
  req_mean_input_handle_time = calc_mean_input_handle_time(router_req_input_payload_tensor, router_req_payload_tensor_copy, req_handle_time)
  resp_mean_input_handle_time = calc_mean_input_handle_time(router_resp_input_payload_tensor, router_resp_payload_tensor_copy, resp_handle_time)
  req_mean_input_queue_depth = calc_mean_input_queue_depth(router_req_input_payload_tensor, req_mean_input_handle_time)
  resp_mean_input_queue_depth = calc_mean_input_queue_depth(router_resp_input_payload_tensor, resp_mean_input_handle_time)
  req_average_latency = calc_average_latency_tensor(routing_tensor_req_dor, req_mean_input_queue_depth, req_mean_input_handle_time)
  resp_average_latency = calc_average_latency_tensor(routing_tensor_resp_dor, resp_mean_input_queue_depth, resp_mean_input_handle_time)
  return (req_average_latency.sum(), resp_average_latency.sum())

In [22]:
pir_list = [0.025,0.050, 0.075, 0.100, 0.125, 0.150, 0.175, 0.200]
# pir_list = [0.05, 0.1]
latencies_req = list()
latencies_resp = list()
latencies = list()
for pir in pir_list:
  (req_lat, resp_lat) = calc_average_latency_pir_pipeline(pir)
  latencies_req.append(req_lat / 2 / (mesh_dim_x + mesh_dim_y - 2) / (mesh_dim_x - 2) / (mesh_dim_y - 2))
  latencies_resp.append(resp_lat / 2 / (mesh_dim_x + mesh_dim_y - 2) / (mesh_dim_x - 2) / (mesh_dim_y - 2))
  latencies.append(latencies_req[-1] + latencies_resp[-1])
  print(f"{pir} done: latency_req = {latencies_req[-1]}; latency_resp = {latencies_resp[-1]}; total = {latencies[-1]}")

Blocking possibility done
Blocking possibility done
Blocking time done
Blocking time done
Request handle time done
Request handle time done
Mean request handle time done
Mean request handle time done
Mean queue depth done
Mean queue depth done
Average latency done
Average latency done
0.025 done: latency_req = 16.708625415094854; latency_resp = 16.713740135452724; total = 33.42236555054758
Blocking possibility done
Blocking possibility done
Blocking time done
Blocking time done
Request handle time done
Request handle time done
Mean request handle time done
Mean request handle time done
Mean queue depth done
Mean queue depth done
Average latency done
Average latency done
0.05 done: latency_req = 16.768510526074685; latency_resp = 16.778096862877888; total = 33.54660738895257
Blocking possibility done
Blocking possibility done
Blocking time done
Blocking time done
Request handle time done
Request handle time done
Mean request handle time done
Mean request handle time done
Mean queue dept

In [23]:
print(latencies)

[33.42236555054758, 33.54660738895257, 33.70327803676123, 33.88976709552704, 34.1037452805231, 34.34324592010164, 34.606728423587654, 34.89313700836439]
